In [1]:
# 08_ablation.ipynb
# Ablations (Reviewer 3): (a) the single-parameter reduction -- losses depend on
# the two dials (PDO, grid g) only through Delta = g ln2 / PDO; (b) the rounding
# rule -- round-to-nearest is unbiased (E[e]=0) while truncation biases the score
# by -P*Delta/2; (c) robustness of the sqrt(P) law to cross-feature correlation.

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"savefig.dpi": 600, "figure.dpi": 110, "axes.grid": True,
                     "grid.color": "0.9", "font.size": 9, "axes.edgecolor": "0.3"})
GREYS = ["0.0", "0.3", "0.5", "0.65"]; LS = ["-", "--", "-.", ":"]; MK = ["o", "s", "^", "D"]
Path("../results/figures").mkdir(parents=True, exist_ok=True)
Path("../results/tables").mkdir(parents=True, exist_ok=True)
def _save(fig, name):
    fig.savefig(f"../results/figures/{name}.png", bbox_inches="tight")
    fig.savefig(f"../results/figures/{name}.pdf", bbox_inches="tight")

df = pd.read_parquet("../data/synthetic_population.parquet")
ccols = [c for c in df.columns if c.startswith("c")]
C = df[ccols].to_numpy(); f = df["f"].to_numpy(); p = df["p"].to_numpy(); y = df["y"].to_numpy()
P = len(ccols); b0 = float((f - C.sum(1)).mean())
def fq(D): return b0 + (D * np.round(C / D)).sum(1)

def migration(fc, s, K):
    q = np.quantile(fc, np.linspace(0, 1, K + 1)[1:-1]); return float(np.mean(np.digitize(fc, q) != np.digitize(s, q)))

# (a) reduction: sweep (PDO, g) and confirm losses collapse onto Delta
rows = []
for PDO in [20, 40, 80]:
    for g in [1, 2, 4]:
        D = g * np.log(2) / PDO
        s = b0 + (D * np.round(C / D)).sum(1)   # equals the PDO points formula up to sign
        rows.append(dict(PDO=PDO, g=g, Delta=D,
                         prob_mad=float(np.abs(1/(1+np.exp(-s)) - p).mean()),
                         migration=migration(f, s, 10)))
red = pd.DataFrame(rows).sort_values("Delta")
red.to_csv("../results/tables/08_reduction.csv", index=False)
spread = red.groupby("Delta")["migration"].agg(lambda x: x.max() - x.min()).max()
print("single-parameter reduction: max migration spread among equal-Delta pairs = %.2e" % spread)

# (b) rounding rule at Delta=0.2
D = 0.2; rule_rows = []
rr = np.random.default_rng(0)
for rule in ["round", "floor", "stochastic"]:
    if rule == "round": q = np.round(C / D)
    elif rule == "floor": q = np.floor(C / D)
    else:
        u = C / D; q = np.floor(u) + (rr.random(C.shape) < (u - np.floor(u)))
    s = b0 + (D * q).sum(1); e = s - f
    rule_rows.append(dict(rule=rule, mean_e=float(e.mean()), std_e=float(e.std()),
                          prob_mad=float(np.abs(1/(1+np.exp(-s)) - p).mean())))
rule_tab = pd.DataFrame(rule_rows); rule_tab.to_csv("../results/tables/08_rounding.csv", index=False)
print(rule_tab.round(4).to_string(index=False))

# (c) cross-feature correlation stress
def gen_corr(rho, Pv=20, N=20000, B=6, seed=1):
    r = np.random.default_rng(seed); gg = r.normal(size=(N, 1))
    z = np.sqrt(rho) * gg + np.sqrt(1 - rho) * r.normal(size=(N, Pv))
    ed = np.quantile(z, np.linspace(0, 1, B + 1), axis=0); bi = np.empty((N, Pv), int)
    for j in range(Pv): bi[:, j] = np.clip(np.digitize(z[:, j], ed[1:-1, j]), 0, B - 1)
    bv = (np.arange(B) - (B - 1) / 2) / ((B - 1) / 2)
    st = np.concatenate([[1.3, 1.1, 0.9, 0.8], r.uniform(0.2, 0.6, Pv - 4)])
    return np.stack([st[j] * 0.45 * bv[bi[:, j]] for j in range(Pv)], axis=1)
rhos = [0.0, 0.3, 0.6, 0.9]; D = 0.1; cr = []
for rho in rhos:
    Cm = gen_corr(rho); e = (D * np.round(Cm / D) - Cm).sum(1)
    cr.append(e.std() / (D * np.sqrt(20 / 12)))
print("correlation stress std(e)/typical:", [round(x, 3) for x in cr])

fig, ax = plt.subplots(2, 2, figsize=(9, 7))
for (PDO, mk, col) in zip([20, 40, 80], MK, GREYS):
    sub = red[red.PDO == PDO]
    ax[0, 0].plot(sub.Delta, sub.migration, mk, color=col, label=f"PDO={PDO}")
gm = red.groupby("Delta").migration.mean()
ax[0, 0].plot(gm.index.values, gm.values, ls=":", color="0.4")
ax[0, 0].set_xscale("log"); ax[0, 0].set_xlabel(r"$\Delta=g\ln2/\mathrm{PDO}$"); ax[0, 0].set_ylabel("migration (K=10)"); ax[0, 0].legend(fontsize=7)
for (PDO, mk, col) in zip([20, 40, 80], MK, GREYS):
    sub = red[red.PDO == PDO]
    ax[0, 1].plot(sub.Delta, sub.prob_mad, mk, color=col, label=f"PDO={PDO}")
ax[0, 1].set_xscale("log"); ax[0, 1].set_yscale("log"); ax[0, 1].set_xlabel(r"$\Delta$"); ax[0, 1].set_ylabel("probability MAD"); ax[0, 1].legend(fontsize=7)
xpos = np.arange(3)
ax[1, 0].bar(xpos - 0.2, rule_tab.mean_e, 0.4, color="0.2", label=r"mean $e$ (bias)")
ax[1, 0].bar(xpos + 0.2, rule_tab.prob_mad, 0.4, color="0.6", label="probability MAD")
ax[1, 0].set_xticks(xpos); ax[1, 0].set_xticklabels(rule_tab.rule); ax[1, 0].set_ylabel(r"value at $\Delta=0.2$"); ax[1, 0].legend(fontsize=7)
ax[1, 1].plot(rhos, cr, "-o", color="0.0")
ax[1, 1].axhspan(0.8, 1.2, color="0.85", label=r"$\pm20\%$ band")
ax[1, 1].axhline(1.0, ls=":", color="0.4")
ax[1, 1].set_xlabel("cross-feature correlation"); ax[1, 1].set_ylabel(r"$\mathrm{std}(e)/\Delta\sqrt{P/12}$"); ax[1, 1].legend(fontsize=7)
fig.tight_layout(); _save(fig, "08_ablation"); plt.close(fig)

single-parameter reduction: max migration spread among equal-Delta pairs = 0.00e+00
      rule  mean_e  std_e  prob_mad
     round -0.0000 0.2831    0.0141
     floor -2.0000 0.2266    0.0657
stochastic -0.0001 0.3748    0.0177
correlation stress std(e)/typical: [np.float64(1.202), np.float64(1.123), np.float64(1.019), np.float64(0.837)]
